# 📋 Ejercicio Clase 3 — PSI, Reservas y Decisión de Re-Entrenamiento
## Diplomado ML en Seguros · Subtema 4

---

### Contexto

Eres actuario en **DirectoCar**, aseguradora de autos. Tienes un modelo GBM
para predecir el costo de siniestros. El modelo fue entrenado con datos de
2019–2021 y lleva 3 años en producción.

Esta clase agrega los conceptos finales del tema:
1. **Deflactar el target** antes de entrenar para separar inflación de drift
2. **PSI** para detectar si el portafolio de producción cambió
3. **Backtesting de reservas** — el error de reserva concreto por año
4. **Matriz de decisión** de re-entrenamiento integrando todo

### Inflación del sector
- Refacciones: **5.5% anual** (acumulado 2019–2024: +30%)
- Mano de obra: **4.0% anual** (acumulado 2019–2024: +22%)

**No cambies `random_state=2024`.**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score

np.random.seed(2024)

INF_REF = 0.055
INF_MO  = 0.040

def idx_inf(anio, base=2019):
    return 0.60*(1+INF_REF)**(anio-base) + 0.40*(1+INF_MO)**(anio-base)

print("Índice de inflación compuesta (2019 = 1.0):")
for a in range(2019,2025):
    print(f"  {a}: {idx_inf(a):.4f} (+{(idx_inf(a)-1)*100:.1f}%)")

---
## Parte 1 — Portafolio DirectoCar 2019–2024 (NO modificar)

In [ ]:
np.random.seed(2024)
ANIOS = list(range(2019,2025))
N_ANIO = {2019:1600,2020:1450,2021:1750,2022:1950,2023:2100,2024:2200}

registros = []
for anio in ANIOS:
    n   = N_ANIO[anio]
    tc  = np.random.choice([0,1,2,3],n,p=[.24,.36,.31,.09])
    zon = np.random.randint(1,6,n)
    av  = np.random.randint(0,21,n)
    cob = np.random.choice([1,2,3],n,p=[.32,.37,.31])
    vv  = np.clip(np.random.lognormal(np.log(260),0.68,n),55,1700).round(1)
    cb  = 11+7*tc+2.8*zon+0.75*av+4.8*cob+0.038*vv+np.abs(np.random.normal(0,16,n))
    cn  = (cb * idx_inf(anio)).round(2)
    for i in range(n):
        registros.append({'anio':anio,'tipo_colision':tc[i],'zona':zon[i],
                          'anios_vehiculo':av[i],'cobertura':cob[i],
                          'valor_vehiculo_k':vv[i],'costo_nominal_k':cn[i],
                          'costo_2019_k':cb[i].round(2),'idx':idx_inf(anio)})

df = pd.DataFrame(registros).sort_values('anio').reset_index(drop=True)
FEATURES = ['tipo_colision','zona','anios_vehiculo','cobertura','valor_vehiculo_k']

print(f"DirectoCar: {len(df):,} siniestros 2019–2024")
for a in ANIOS:
    s = df[df.anio==a]
    print(f"  {a}: n={len(s):,}  nominal=${s.costo_nominal_k.mean():.1f}K  "
          f"deflact=${s.costo_2019_k.mean():.1f}K")

---
## Parte 2 — Separar inflación de drift

### 🔧 2.1 — Walk-Forward con costo nominal y deflactado

In [ ]:
# 🔧 COMPLETA: Walk-Forward para 2022, 2023, 2024
# Dos versiones del target: costo nominal Y costo deflactado a 2019
# Para cada versión: MAE y R²

ANIOS_TEST = [2022, 2023, 2024]
res_nom, res_def = [], []

# K-Fold nominal (INCORRECTO)
kf = KFold(n_splits=5, shuffle=True, random_state=2024)
pipe_base = Pipeline([('sc',StandardScaler()),('m',GradientBoostingRegressor(
    n_estimators=150,max_depth=4,learning_rate=0.08,random_state=2024))])
mae_kf = -cross_val_score(pipe_base, df[FEATURES].values, df['costo_nominal_k'].values,
                           cv=kf, scoring='neg_mean_absolute_error').mean()
print(f"K-Fold nominal (INCORRECTO): MAE = ${mae_kf:,.2f}K")
print()

print(f"  {'Año':>5}  {'MAE Nominal':>13}  {'MAE Deflact':>13}  "
      f"{'R² Nom':>8}  {'R² Def':>8}  {'Diagnóstico'}")
print("-"*80)

mae_def_base = None   # referencia: MAE deflactado del primer año de test

for anio_test in ANIOS_TEST:
    mtr = df['anio'] < anio_test; mte = df['anio'] == anio_test
    X_tr = df.loc[mtr,FEATURES].values; X_te = df.loc[mte,FEATURES].values

    # 🔧 COMPLETA: entrena y evalúa con costo NOMINAL
    # --- TU CÓDIGO AQUÍ ---
    mae_nom, r2_nom, pred_nom = ..., ..., ...   # (MAE, R², predicciones sobre X_te)

    # 🔧 COMPLETA: entrena y evalúa con costo DEFLACTADO
    # Recuerda: el target de entrenamiento es df.loc[mtr,'costo_2019_k']
    # y el target de evaluación es df.loc[mte,'costo_2019_k']
    # --- TU CÓDIGO AQUÍ ---
    mae_def, r2_def, pred_def = ..., ..., ...   # (MAE, R², predicciones en pesos 2019)

    # Diagnóstico correcto: comparar vs el año BASE (2022), no vs el anterior
    if mae_def_base is None:
        mae_def_base = mae_def
        diag = "Año base de referencia"
    elif abs(mae_def - mae_def_base) < 2.0:
        diag = "✅ Inflación, no drift del modelo"
    else:
        diag = f"⚠️ Drift real (subió ${mae_def-mae_def_base:.1f}K vs base)"

    res_nom.append({'anio':anio_test,'mae_nom':mae_nom,'r2_nom':r2_nom})
    res_def.append({'anio':anio_test,'mae_def':mae_def,'r2_def':r2_def,
                    'pred_def':pred_def,                         # predicciones en pesos 2019
                    'y_te_nom':df.loc[mte,'costo_nominal_k'].values,
                    'y_te_def':df.loc[mte,'costo_2019_k'].values})

    print(f"  {anio_test:>5}  ${mae_nom:>11,.2f}K  ${mae_def:>11,.2f}K  "
          f"{r2_nom:>8.4f}  {r2_def:>8.4f}  {diag}")

print()
print("¿El MAE nominal sube? ¿El MAE deflactado se mantiene estable?")
print("Esa es la clave: si el deflactado es estable, el modelo funciona bien — solo es inflación.")


---
## Parte 3 — PSI: ¿el portafolio de producción cambió?

### 🔧 3.1 — Calcula el PSI por variable

In [ ]:
# 🔧 COMPLETA: implementa la función calcular_psi y úsala
# para calcular el PSI de cada variable en FEATURES
# Base = 2019–2021, comparar contra 2022, 2023, 2024

def calcular_psi(base_values, actual_values, n_bins=10):
    """
    Calcula el PSI para comparar distribuciones.
    PSI < 0.10 → estable
    PSI 0.10–0.25 → monitorear
    PSI > 0.25 → cambio significativo
    """
    # --- TU CÓDIGO AQUÍ ---
    # Pista: usar np.histogram con bins en percentiles de base_values
    psi = ...
    return psi

base_mask = df['anio'].isin([2019,2020,2021])

print("PSI por variable (base: 2019–2021):")
print()
print(f"  {'Variable':>20}  {'2022':>8}  {'2023':>8}  {'2024':>8}  {'Diagnóstico'}")
print("-"*68)

for feat in FEATURES:
    base_vals = df.loc[base_mask, feat].values
    psies = []
    for a in [2022,2023,2024]:
        act_vals = df.loc[df['anio']==a, feat].values
        psies.append(calcular_psi(base_vals, act_vals))
    max_p = max(psies)
    diag = "✅ Estable" if max_p < 0.10 else ("⚠️ Monitorear" if max_p < 0.25 else "🔴 Cambio")
    print(f"  {feat:>20}  "+"  ".join(f"{p:>8.4f}" for p in psies)+f"  {diag}")

---
## Parte 4 — Backtesting de reservas

### 🔧 4.1 — Error de reserva por año

In [ ]:
# 🔧 COMPLETA: backtesting de reservas
# Para cada año de test:
#   reserva_modelo  = media(pred_def) × idx_inf(año)
#                     → pred_def está en pesos 2019; multiplicar por índice reinflata al año real
#   reserva_base    = media de costos nominales de los 3 años anteriores (sin modelo)
#   costo_real      = media de costos nominales reales del año de test
#   error           = reserva − costo real
#                     (+) sobre-reserva  (−) sub-reserva

print("BACKTESTING DE RESERVAS — error promedio por siniestro:")
print()
print(f"  {'Año':>5}  {'Reserva modelo':>16}  {'Reserva base':>14}  "
      f"{'Costo real':>12}  {'Error modelo':>14}  {'Error base':>12}")
print("-"*82)

for r in res_def:
    a = r['anio']

    # 🔧 COMPLETA
    reserva_modelo = r['pred_def'].mean() * idx_inf(a)   # reinflatar a pesos del año
    reserva_base   = df[df['anio'].isin([a-3,a-2,a-1])]['costo_nominal_k'].mean()
    costo_real     = r['y_te_nom'].mean()

    err_mod  = reserva_modelo - costo_real
    err_base = reserva_base   - costo_real

    print(f"  {a:>5}  ${reserva_modelo:>14.2f}K  ${reserva_base:>12.2f}K  "
          f"${costo_real:>10.2f}K  ${err_mod:>+12.2f}K  ${err_base:>+10.2f}K")

print()
print("(+) = sobre-reserva  ·  (−) = sub-reserva (riesgo regulatorio)")
print()
print("¿El modelo produce errores menores que el baseline?")
print("¿La dirección del error (+ o −) es consistente año a año?")


---
## Parte 5 — Matriz de decisión de re-entrenamiento

### 🔧 5.1 — Gráfica con los 4 paneles del diagnóstico

In [ ]:
# 🔧 COMPLETA: figura con 4 paneles (o los que consideres necesarios)
# 1. MAE nominal vs deflactado por año
# 2. R² nominal vs deflactado por año
# 3. PSI de la variable más importante por año
# 4. Error de reserva (modelo vs baseline) por año

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Diagnóstico Completo — DirectoCar (Clase 3)', fontweight='bold')

# Panel 1
ax = axes[0,0]
# --- TU CÓDIGO AQUÍ ---

# Panel 2
ax = axes[0,1]
# --- TU CÓDIGO AQUÍ ---

# Panel 3
ax = axes[1,0]
# --- TU CÓDIGO AQUÍ ---

# Panel 4
ax = axes[1,1]
# --- TU CÓDIGO AQUÍ ---

plt.tight_layout()
plt.savefig('ej3_costo.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Parte 6 — La decisión final de re-entrenamiento

### 🔧 6.1 — Aplica la matriz de decisión

In [ ]:
# 🔧 COMPLETA: con base en tu análisis, llena la siguiente tabla
# para el año 2024 (el más reciente)

anio_eval = 2024
r_nom_24 = [r for r in res_nom if r['anio']==anio_eval][0]
r_def_24 = [r for r in res_def if r['anio']==anio_eval][0]

# ¿El MAE deflactado subió más de $2K vs el año anterior?
mae_def_2023 = [r for r in res_def if r['anio']==2023][0]['mae_def']
mae_def_2024 = r_def_24['mae_def']
drift_real = mae_def_2024 - mae_def_2023 > 2.0

# ¿Alguna variable tiene PSI > 0.10 en 2024?
base_vals_vv = df.loc[df['anio'].isin([2019,2020,2021]), 'valor_vehiculo_k'].values
act_vals_vv  = df.loc[df['anio']==2024, 'valor_vehiculo_k'].values
psi_critico  = calcular_psi(base_vals_vv, act_vals_vv) > 0.10   # usa la variable más importante

print(f"DIAGNÓSTICO PARA {anio_eval}:")
print()
print(f"  MAE deflactado 2023: ${mae_def_2023:.2f}K")
print(f"  MAE deflactado 2024: ${mae_def_2024:.2f}K")
print(f"  ¿Hay drift real (subió > $2K)? {'SÍ ⚠️' if drift_real else 'NO ✅'}")
print()
print(f"  ¿PSI crítico (valor_vehiculo_k > 0.10)? {'SÍ ⚠️' if psi_critico else 'NO ✅'}")
print()

# 🔧 COMPLETA: con base en los valores anteriores, determina la acción
if not drift_real and not psi_critico:
    accion = "✅ No re-entrenar. El modelo está bien. Es solo inflación."
elif drift_real and not psi_critico:
    accion = "⚠️ Investigar. Hay drift del modelo pero el portafolio no cambió."
elif not drift_real and psi_critico:
    accion = "🔧 Investigar variables. El portafolio cambió pero el modelo aguanta."
else:
    accion = "🔴 Re-entrenar. Hay drift del modelo Y el portafolio cambió."

print(f"  ACCIÓN RECOMENDADA: {accion}")

---
## Preguntas de reflexión

### 📝 Preguntas

**a)** El MAE nominal sube de $9.7K (2022) a $12.4K (2024), pero el MAE deflactado
se mantiene en ~$7.8K. ¿Qué le dirías al director de Reservas cuando pregunta
"¿por qué el modelo se equivoca más que antes?"

> _Tu respuesta aquí_

---

**b)** El PSI de todas las variables es < 0.01 en 2024 — muy estable.
Si el portafolio NO cambió y el MAE deflactado tampoco cambió,
¿qué causa el aumento del MAE nominal? ¿Qué implicación tiene eso
para la decisión de re-entrenamiento?

> _Tu respuesta aquí_

---

**c)** El backtesting de reservas muestra que el modelo produce errores
de ~$0.3K por siniestro, mientras el baseline produce ~$7K.
¿Cómo traducirías esa diferencia al director financiero en términos
del costo de capital que se libera al usar el modelo?
(Supón 80,000 siniestros al año y costo de capital del 12% anual.)

> _Tu respuesta aquí_

---

**d) BONUS — El índice como variable predictora:**
Agrega `idx` (índice de inflación acumulada) como variable al modelo nominal.
¿Mejora el MAE nominal? ¿Por qué? ¿Qué riesgo tiene en producción si el índice
del siguiente año no se conoce exactamente al momento de reservar?

```python
# 🔧 BONUS
FEATURES_IDX = FEATURES + ['idx']
# Entrena Walk-Forward con nominal usando FEATURES_IDX
# Compara MAE nominal con y sin el índice
# --- TU CÓDIGO AQUÍ ---
```
